In [5]:
import dash
import dash_table
from dash import html, dcc, Input, Output, callback
import dash_bootstrap_components as dbc
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
from dash.exceptions import PreventUpdate

#Importation base et traitement#
df = pd.read_csv("./supermarket_sales.csv", index_col=0)
df['Date'] = pd.to_datetime(df['Date'])
df['Mois'] = df['Date'].dt.month
df = df[['City', 'Gender', 'Total','Product line', 'Rating', 'Date','Mois']]

#Calcul motant total des achats#
def Montant_tot_achats(data):
    return data['Total'].sum()

#Somme total motant d'achat par genre et par ville#
def Sum_montant_tot(data, top=10, ascending=False):
    resultat = pd.crosstab(
        [data['Gender'], data['City']], 
        'vente', 
        values=data['Total'], 
        aggfunc= sum, 
        rownames=['Sexe', 'Ville'],
        colnames=['']
    ).reset_index().groupby(
        ['Sexe'], as_index=False, group_keys=True
    ).apply(
        lambda x: x.sort_values('vente', ascending=ascending).iloc[:top, :]
    ).reset_index(drop=True).set_index(['Sexe', 'Ville'])

    return resultat

#Somme total motant d'achat par ville#
def Sum_montant_tot2(data, top=10, ascending=False):
    resultat = pd.crosstab(
        data['City'],
        'vente', 
        values=data['Total'], 
        aggfunc=sum, 
        rownames=['Ville'],
        colnames=['']
    ).reset_index().groupby(
        ['Ville'], as_index=False, group_keys=True
    ).apply(
        lambda x: x.sort_values('vente', ascending=ascending).iloc[:top, :]
    ).reset_index(drop=True).set_index(['Ville'])

    return resultat



#Histogramme de la répartition des montants totaux des achats par sexe et par ville#
def Histogramme(data) :
    df_plot = Sum_montant_tot(data, ascending=False)
    pantone_colors = {
        'Male': '#009473',  
        'Female': '#dfcb00'
    }
    graph = px.bar(
        df_plot,
        y='vente', 
        x=df_plot.index.get_level_values(1),
        color=df_plot.index.get_level_values(0), 
        barmode='group',
        title="Montant total des achats par sexe et ville",
        labels={"vente": "Montant total des ventes", "x": "Ville", "color": "Genre"},
        width=600, height=400,
        color_discrete_map=pantone_colors,
        orientation='v'
    ).update_layout(
        plot_bgcolor='rgba(0,0,0,0)'
    )
    return graph

#Evolution du montant total des achats par genre et par ville#
def Evolution_motant_tot(data):
    df_plot = data.groupby(pd.Grouper(key='Date', freq='W')).apply(Sum_montant_tot2)[:-1]
    df_plot_reset = df_plot.reset_index()
    pantone_colors2 = {
        'Mandalay': '#9b1b30',  
        'Naypyitaw': '#0f4c81',
        'Yangon' : '#88b04b'
    }
    chiffre_evolution = px.line(
        df_plot_reset, x='Date', y='vente', color='Ville',
        title="Evolution du montant des achats par semaine",
        labels={"Semaine": "Semaine", "vente": "Montant total des achats"},
        color_discrete_map=pantone_colors2
    )
    
    chiffre_evolution.update_layout( 
        width=800, height=400,
        margin=dict(r=100, t=60, b=0),
    ).update_layout(
        plot_bgcolor='rgba(0,0,0,0)'
    )
    return chiffre_evolution

#Diagramme circulaire de la répartitions de la catégorie de produit par genre et par ville#
def Repartition_produits_par_sexe_et_ville(data):
    total_achats = len(data)
    df_plot = data.groupby(['Gender', 'City', 'Product line']).size().reset_index(name='Frequency')
    df_plot['Percentage'] = (df_plot['Frequency'] / total_achats) * 100
    pantone_colors = {
        'Male': '#009473',  
        'Female': '#dfcb00'
    }
    fig = px.sunburst(df_plot,
                      path=['Gender', 'City', 'Product line'],
                      values='Percentage',
                      color='Gender',
                      color_discrete_map=pantone_colors,
                      title="Répartition des catégories de produits par sexe et par ville",
                      labels={"City": "Ville", "Gender": "Genre", "Product line": "Catégorie de produit", "Percentage": "Pourcentage"},
                      width=600, height=400)

    return fig

#Indicateur 1 et 2#
def create_card(title, content, color="#0f4c81"):
    card_content = [
        dbc.CardHeader(title, className="text-center"),
        dbc.CardBody(
            [
                html.H5(content, className="card-title"),
            ],
            className="text-center"
        )
    ]
    card = dbc.Card(card_content, color='#0f4c81', inverse=True)
    return card




#DASH#
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])
colors = {
    'background': '#cbb1b4',
    'background2': '#f2e9ea'
}

app.layout = dbc.Container([
    dbc.Row([
        dbc.Col([html.Span("\u200B")], width=4),
        dbc.Col(html.Div(html.Strong("Analyse conjoncturel des ventes réalisées"), style={"text-align": "center", "line-height": "100px", "font-size": "20px"}), width=4),
        dbc.Col([
            dbc.Row(["\u200B"],style={"background-color": colors['background']}),
            dbc.Row([
                dcc.Dropdown(
                id ='city',
                options= [{'label':city, 'value':city} for city in df['City'].dropna().unique()],
                multi=True,
                searchable=True,
                placeholder='Selection : ville')
            ]),
                dbc.Row([
        dcc.Dropdown(
            id='gender',
            options=[{'label': gender, 'value': gender} for gender in df['Gender'].dropna().unique()],
            multi=True,
            searchable=True,
            placeholder='Sélectionnez le genre')
            ]),
        ], width=4)],style={"background-color": colors['background']}),
    dbc.Row([
        dbc.Col(
            html.Div([
                dbc.Row([
                    dbc.Col(width=3),
                    dbc.Col(width=6, children=dcc.Graph(id='Graphique-1'), style={'background-color':colors['background2']}),
                    dbc.Col(width=3)],style={"background-color": colors['background2']}),
                html.Br(style={"background-color": colors['background2']}),
                dbc.Row([
                    dbc.Col(html.Div(id='Indicateur-1'), width=6),
                    dbc.Col(html.Div(id='Indicateur-2'), width=6)],style={"background-color": colors['background2']})
        ]), width=12)],style={"background-color": colors['background2']}),
    dbc.Row(["\u200B"],style={"background-color": colors['background2']}),
    dbc.Row([
        dbc.Col(html.Div([html.Div([dcc.Graph(id='Histogramme-1')])]), width=5, style={'background-color':colors['background2']}),
        dbc.Col(width=2, style={"background-color": colors['background2']}),
        dbc.Col(html.Div([html.Div([dcc.Graph(id='Circulaire-1')])]), width=5, style={'background-color':colors['background2']})
    ],style={"background-color": colors['background2']})
], fluid = True)

@callback(
    Output('Graphique-1', 'figure'),
    [Input('city', 'value'),Input('gender', 'value')]
)
def Evolution_montant(city, gender):
    if city and not gender:
        df_temps = df[df['City'].isin(city)]
    elif gender and not city:
        df_temps = df[df['Gender'].isin(gender)]
    elif city and gender:
        df_temps = df[df['Gender'].isin(gender) & df['City'].isin(city)]
    else:
        df_temps = df
    return Evolution_motant_tot(df_temps)

@callback(
    Output('Histogramme-1', 'figure'),
    [Input('city', 'value'),Input('gender', 'value')]
)
def Histogramme2(city, gender):
    if city and not gender:
        df_temps = df[df['City'].isin(city)]
    elif gender and not city:
        df_temps = df[df['Gender'].isin(gender)]
    elif city and gender:
        df_temps = df[df['Gender'].isin(gender) & df['City'].isin(city)]
    else:
        df_temps = df
    return Histogramme(df_temps)


@callback(
    Output('Circulaire-1', 'figure'),
    [Input('city', 'value'),Input('gender', 'value')]
)
def Circulaire(city, gender):
    if city and not gender:
        df_temps = df[df['City'].isin(city)]
    elif gender and not city:
        df_temps = df[df['Gender'].isin(gender)]
    elif city and gender:
        df_temps = df[df['Gender'].isin(gender) & df['City'].isin(city)]
    else:
        df_temps = df
    return Repartition_produits_par_sexe_et_ville(df_temps)

@app.callback(
    Output('Indicateur-1', 'children'),
    [Input('gender', 'value'), Input('city', 'value')]
)
def update_indicator_1(gender, city):
    if not gender or not city:
        raise PreventUpdate

    filtered_df = df[df['Gender'].isin(gender) & df['City'].isin(city)]
    total_sales = filtered_df['Total'].sum()
    total_sales_card = create_card("Total des Ventes", f"{total_sales:,.2f}€", "success")
    
    return total_sales_card

@app.callback(
    Output('Indicateur-2', 'children'),
    [Input('gender', 'value'), Input('city', 'value')]
)
def update_indicator_2(gender, city):
    if not gender or not city:
        raise PreventUpdate
    filtered_df = df[df['Gender'].isin(gender) & df['City'].isin(city)]
    average_rating = filtered_df['Rating'].mean()
    average_rating_card = create_card("Évaluation Moyenne", f"{average_rating:.1f}/10", "warning")
    
    return average_rating_card





if __name__ == '__main__':
    app.run_server(debug=True, port=8054, jupyter_mode="external")





Dash app running on http://127.0.0.1:8054/


/Users/rododo/opt/anaconda3/lib/python3.9/site-packages/plotly/express/_core.py:1637: FutureWarning:

The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.

/Users/rododo/opt/anaconda3/lib/python3.9/site-packages/plotly/express/_core.py:1637: FutureWarning:

The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.

/Users/rododo/opt/anaconda3/lib/python3.9/site-packages/plotly/express/_core.py:1637: FutureWarning:

The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.

/Users/rododo/opt/anaconda3/lib/python3.9/site-packages/plotly/express/_core.py:1637: FutureWarning:

The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.

/Users/rododo/opt/anaconda3/lib/python3.9/site-packages/plotly/express/_core.py:1637: FutureWarning:

The frame.appe